# Hanogt AI — Kaggle'da sürekli eğitim

Bu defter Hanogt AI'ın dil modelini (**Qwen3-8B**, QLoRA) Kaggle'ın ücretsiz GPU'suyla **tur tur** eğitir.
Her çalıştırmada en güncel eğitim kodunu ve veri kaydını GitHub'dan alır; bu yüzden yeni veri setleri
eklendikçe defteri yeniden yüklemen gerekmez, yalnızca yeniden çalıştırırsın.

**Ne yapar?**
1. **Veri:** Kayıtta (`training/sources.json`) yeni ya da değişmiş kaynak varsa yalnızca onları Hugging Face'ten indirir,
   Hanogt'un kendi bilgisiyle ve GitHub kaynaklarıyla karıştırır, senin **özel** veri deponda (`DATA_REPO`) saklar.
2. **Eğitim (GPU varsa):** Yarım kalan tur varsa kaldığı yerden sürdürür; yoksa yeni turu seçer
   (henüz öğrenilmemiş veri + unutmamak için eskilerden tekrar + Hanogt'un kendi örnekleri) ve bir önceki turun
   üzerine eğitir. Sonuç senin **özel** model deponda (`MODEL_REPO`) durur.

### Bir kez yapılacaklar
1. **Telefon doğrulaması:** kaggle.com → profil resmi → *Settings* → *Phone Verification*. Bu olmadan GPU ve internet açılmaz.
2. **Token:** üst menü → *Add-ons* → *Secrets* → *Add Secret* → Label: `HF_TOKEN`, Value: Hugging Face **Write** token'ın.
   Kaydet ve bu deftere bağlı olduğunu gösteren kutuyu işaretle. Token'ı asla bir hücreye yazma.
3. **Oturum ayarları** (sağ panel → *Session options*): *Internet*: **On**.

### Çalıştırma
- **İlk kez (önerilen):** *Accelerator*: **None** seçip *Save Version → Save & Run All (Commit)*. Veri hazırlanır,
  GPU kotan harcanmaz (ilk seferde 2–4 saat sürebilir; sonrakilerde yalnızca değişen kaynaklar indirilir).
- **Eğitim:** *Accelerator*: **GPU T4 x2** seçip yine *Save Version → Save & Run All (Commit)*. Tarayıcıyı kapatabilirsin;
  en fazla ~11 saat çalışır ve kendisi durur.
- **Sürekli eğitim:** Haftada bir (Kaggle haftada ~30 saat GPU verir) yeniden *Save & Run All* yap. Her çalıştırma bir
  sonraki tura geçer ya da yarım kalanı bitirir. Yeni veri setleri eklendikçe kendiliğinden onları da öğrenir.


In [ ]:
# Ayarlar: depolar senin hesabında ve özel.
REPO = "https://github.com/Hanstudios1/HanogtCodev.git"  # eğitim kodu ve veri kaydı
BRANCH = "main"
DATA_REPO = "HanStudios/hanogt-sft-pool"                 # veri havuzu ve turların verisi
MODEL_REPO = "HanStudios/hanogt-ai-qwen3-8b-lora"        # eğitilen bağdaştırıcı ve eğitim durumu
BASE_MODEL = "Qwen/Qwen3-8B"
ROUND_SAMPLES = 20000   # bir turdaki örnek sayısı (2× T4'te kabaca 10–20 saat; oturumlara bölünür)
HOURS = 11.0            # Kaggle oturumu 12 saat: kayıt ve yükleme için 1 saat pay


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN hazır (değeri yazdırılmaz)")


In [ ]:
# En güncel eğitim kodu ve gerekli paketler.
!rm -rf /tmp/hanogt-code && git clone -q --depth 1 --branch {BRANCH} {REPO} /tmp/hanogt-code && git -C /tmp/hanogt-code log -1 --format="kod: %h %s"
!pip install -q -U "datasets>=3.0" "huggingface_hub>=0.35" "transformers>=4.56" "peft>=0.17" "accelerate>=1.2" "bitsandbytes>=0.45" "safetensors>=0.4" pytest


In [ ]:
import subprocess
import sys

command = [
    sys.executable, "/tmp/hanogt-code/training/kaggle_run.py",
    "--data-repo", DATA_REPO,
    "--model-repo", MODEL_REPO,
    "--base-model", BASE_MODEL,
    "--round-samples", str(ROUND_SAMPLES),
    "--hours", str(HOURS),
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
if process.wait():
    raise SystemExit(f"Eğitim betiği {process.returncode} koduyla durdu; yukarıdaki son satırlara bak.")


### Sonra
- **Çıktıda "Tur N bitti"** yazıyorsa bağdaştırıcı `MODEL_REPO`'nun kökünde ve `rounds/round-00N/` altında durur;
  eğitim durumu `state/rounds.json` dosyasındadır.
- **"Süre doldu"** yazıyorsa sonraki *Save & Run All* aynı turu kaldığı yerden sürdürür.
- **"Bütün yeni veriler öğrenildi"** yazıyorsa havuz bitti: Claude Code oturumunda yeni veri setleri eklenince yeni turlar başlar.
- Claude Code oturumunda **"Kaggle eğitimi bitti"** de; değerlendirmeyi ve yayına hazırlığı ben yaparım.
